In [ ]:
import matplotlib.pyplot as plt
import os
import scipy.io as scio
import numpy as np
import mne
from antio import read_cnt
from antio.parser import read_triggers
from matplotlib import pyplot as plt
from mne.datasets import testing
from mne.io import read_raw_brainvision
from mne.io import read_raw_ant
from mne.viz import plot_topomap
import glob
from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from mne.stats import f_threshold_mway_rm, f_mway_rm, fdr_correction
import matplotlib.pyplot as plt
import matplotlib
from mne.viz import (plot_ch_adjacency as plot_adj,
                     plot_topomap as plot_topo)

from scipy.signal import detrend

from mne.viz import set_browser_backend
set_browser_backend("qt")

from scipy import stats
from scipy.stats import ttest_rel as ttest
#from scipy.stats import ttest_ind as ttest

from EEG_MVPA_Package import (
    erp_loader,
    pol_to_car,
    multi_window_topo_plot,
    edit_colomap
)

from mne_connectivity import spectral_connectivity_epochs as compute_plv
from mne_connectivity.viz import plot_sensors_connectivity
import pickle
from statsmodels.sandbox.stats.multicomp import multipletests

In [ ]:
def load_catch_index(file_path):
    file = scio.loadmat(file_path)
    stim_index = file['participant_log']['stim_index'][0][0]
    pre_index = stim_index[stim_index[:,4]==1,3]
    train_index = stim_index[stim_index[:,4]==2,3]
    post_index = stim_index[stim_index[:,4]==3,3]
    return pre_index,train_index,post_index

def get_epoch_path(root_path,sub_index,blk,type):
    participant_list = os.listdir(root_path)
    #data_path = glob.glob(root_path+participant_list[sub_index]+'/'+'*'+blk+'_'+type+'_onsetshifting_nondetrend'+'-epo.fif')[0]
    data_path = glob.glob(root_path+participant_list[sub_index]+'/'+'*'+blk+'_'+type+'_onsetshifting'+'-epo.fif')[0]
    #data_path = glob.glob(root_path+participant_list[sub_index]+'/'+'*'+blk+'_'+type+'_nonshifting'+'-epo.fif')[0]
    #data_path = glob.glob(root_path+participant_list[sub_index]+'/'+'*'+blk+'_'+type+'_'+'-epo.fif')[0]
    return data_path

def get_eeg_files_in_folder(path,target):
    files = os.listdir(path)
    target_files = [file for file in files if file.endswith(target)]
    return target_files

def eeg_path(root_path,sub_index):
    participant_list = os.listdir(root_path)
    cnt_path = glob.glob(root_path+participant_list[sub_index]+'/'+'*.cnt')[0]
    evt_path = glob.glob(root_path+participant_list[sub_index]+'/'+'*.evt')[0]
    trg_path = glob.glob(root_path+participant_list[sub_index]+'/'+'*.trg')[0]
    return cnt_path,evt_path,trg_path

def eeg_path_BV(root_path,sub_index):
    participant_list = os.listdir(root_path)
    eeg_path = glob.glob(root_path+participant_list[sub_index]+'/'+'*.eeg')[0]
    vhdr_path = glob.glob(root_path+participant_list[sub_index]+'/'+'*.vhdr')[0]
    vmrk_path = glob.glob(root_path+participant_list[sub_index]+'/'+'*.vmrk')[0]
    return eeg_path,vhdr_path,vmrk_path

def split_session_events(events,event_id):
    button_index = event_id['16']
    sound_index = event_id['96']
    event_list = np.array(events)
    if '64' in event_id:
        event_list[event_list[:,2]==event_id['64'],2]=event_id['96']
    #print(event_list[:,2],'\n',event_id['96'],event_id)
    locker_identifier_1 = [button_index]*2+[sound_index]*3+[button_index]*2+[sound_index]+[button_index]*2+[sound_index]*3+[button_index]*2
    locker_identifier_2 = [button_index]*2+[sound_index]*3+[button_index]*2+[sound_index]+[sound_index]*3+[button_index]*2
    head_trigger_type = [sound_index]*3+[button_index]
    tail_trigger_type = [sound_index]+[button_index]*2+[sound_index]

    locker_head = []
    locker_tail = []
    for i in range(event_list.shape[0]-25):
        if (list(event_list[i:(i+len(locker_identifier_1)+len(head_trigger_type)),2])==locker_identifier_1+head_trigger_type)| \
            (list(event_list[i:(i+len(locker_identifier_2)+len(head_trigger_type)),2])==locker_identifier_2+head_trigger_type):
            locker_head = locker_head + [i+2]
        elif (list(event_list[i:(i+len(locker_identifier_1)+len(tail_trigger_type)),2])==locker_identifier_1+tail_trigger_type)| \
            (list(event_list[i:(i+len(locker_identifier_2)+len(tail_trigger_type)),2])==locker_identifier_2+tail_trigger_type):
            locker_tail = locker_tail + [i+12]
    #print('Head: ',len(locker_head),'.  Tail: ',len(locker_tail))

    arg_sound = np.where(event_list[:,2]==sound_index)[0]

    head = locker_head[0]
    #if len(locker_head)>0:
    #    head = locker_head[0]
    #elif len(locker_head)==0:
    #    head = arg_sound[60]

    if len(locker_tail)>0:
        tail = locker_tail[-1]
    elif len(locker_tail)==0:
        tail = arg_sound[np.where(arg_sound==head)[0][0]+420]-1
    train_event = event_list[(head):(tail+1),:]
    
    #print('Head: ',head,'.  Tail: ',tail)
    #plt.scatter(np.arange(len(event_list[(tail-20):(tail+20),2])),event_list[(tail-20):(tail+20),2])
    #plt.scatter(20,event_list[tail,2],c='r')
    #plt.plot(np.arange(len(event_list[(tail-20):(tail+20),2])),event_list[(tail-20):(tail+20),2])

    #print('Head: ',head,'.  Tail: ',tail)
    #plt.scatter(np.arange(len(event_list[(head-20):(head+20),2])),event_list[(head-20):(head+20),2])
    #plt.scatter(20,event_list[head,2],c='r')
    #plt.plot(np.arange(len(event_list[(head-20):(head+20),2])),event_list[(head-20):(head+20),2])
    
    train_sound_event = train_event[train_event[:,2]==sound_index,:]
    
    if train_sound_event.shape[0]!=420:
        raise ValueError("\nIncorrect training session trial number!!!\n")
    else:
        pre_event = event_list[:(head-1),:]
        post_event = event_list[(tail+1):,:]
        pre_sound_event = pre_event[pre_event[:,2]==sound_index,:]
        if pre_sound_event.shape[0]>60:
            pre_sound_event = pre_sound_event[-61:-1,:]
        post_sound_event = post_event[post_event[:,2]==sound_index,:]
        print('Train_sound_event shape: ',train_sound_event.shape,
              ';   Pre_sound_event shape: ',pre_sound_event.shape,
              ';   Post_sound_event shape: ',post_sound_event.shape)
        return train_sound_event,pre_sound_event,post_sound_event

    #return train_sound_event,head,tail

def split_train_triggers(train_sound_event):
    if train_sound_event.shape[0]!=420:
        raise ValueError("\nIncorrect training session trial number!!!\n")
    else:
        train_triggers = []
        small_block = [1,1,1,3,2,2,2]
        for i in range(60):
            train_triggers = train_triggers + small_block
        event_id = {'SWS_Pre': 5, 'SWS_Post': 6, 'NS': 7}
        train_triggers = np.array(train_triggers)+4
        train_triggers = np.vstack([train_sound_event[:,:2].T,train_triggers])
        return train_triggers.T,event_id

def reject_crazy_epoch(epoch,thr=0.0002):
    data = epoch.get_data()
    epoch1 = epoch.copy()
    delete_list = []
    for i in range(data.shape[0]):
        if np.max(np.abs(data[i,:,:]))>thr:
            delete_list = delete_list + [i]
    delete_list = np.array(delete_list)
    if len(delete_list)>0:
        data = np.delete(data,delete_list,axis=0)
    new_epoch = mne.EpochsArray(data, info=epoch1.info,tmin=epoch1.tmin,baseline=epoch1.baseline)
    return new_epoch

def generate_plot_ch_mask(plot_ch,ch_num,data_length):
    plot_ch_mask = np.zeros([ch_num,data_length])*np.nan
    for i in range(plot_ch_mask.shape[1]):
        plot_ch_mask[plot_ch,i]=np.ones(len(plot_ch))
        
    return plot_ch_mask

In [ ]:
participant_index = 0

In [ ]:
participant_log_path = 'Behaviour'
log_file_list = os.listdir(participant_log_path)
pre_index,train_index,post_index = load_catch_index(participant_log_path+'/'+log_file_list[participant_index])
print(train_index)

In [ ]:
blk_list = ['blk-1','blk-2','blk-3']
type_list = ['lowband','highband']

raw_eeg_path = 'Data/'

epoch_file_path = get_epoch_path(raw_eeg_path,participant_index,blk_list[1],type_list[0])

print(epoch_file_path)

In [ ]:
epoch = mne.read_epochs(epoch_file_path)
print('\n\n\n',epoch.info,'\n\n\n')
print(len(epoch),epoch.baseline,epoch.tmin)

In [ ]:
#epoch.plot(n_epochs=20,n_channels=63,scalings=2e-5)

In [ ]:
participants_index = np.arange(19)
#participants_index = list(set([1,2,3,7,8,9,10,11]) & set([0, 1, 2, 4, 5, 7, 8, 9, 10]))
n_participants = len(participants_index)
print(n_participants)

In [ ]:
resample_freq = 500

In [ ]:
t_window = [0,0.8]
window = np.arange(int(resample_freq*(t_window[0]+0.2)),int(500*(t_window[1]+0.2)))

In [ ]:
reject_thr = 0.0001

In [ ]:
fig_exp_path = 'Raw_fig/'

# High band TFR

In [ ]:
pre_evk_list = []
post_evk_list = []
pre_evk_list_c1 = []
post_evk_list_c1 = []
pre_evk_list_c2 = []
post_evk_list_c2 = []

epoch_pre_early_list = []
epoch_pre_late_list = []
epoch_post_early_list = []
epoch_post_late_list = []

evk_pre_early_list = []
evk_pre_late_list = []
evk_post_early_list = []
evk_post_late_list = []

for i in participants_index:
    epoch_file_path = get_epoch_path(raw_eeg_path,i,blk_list[1],type_list[1])
    epoch_all = mne.read_epochs(epoch_file_path)
    epoch_all.apply_baseline()
    _,train_index,_ = load_catch_index(participant_log_path+'/'+log_file_list[i])
    epoch = epoch_all[train_index==0].resample(sfreq=resample_freq)
    epoch_c1 = epoch_all[train_index==1]
    epoch_c2 = epoch_all[train_index==2]

    epoch_pre_early = reject_crazy_epoch(epoch['SWS_Pre'][:int(len(epoch)/7*3/2)],thr=reject_thr)
    epoch_pre_late = reject_crazy_epoch(epoch['SWS_Pre'][int(len(epoch)/7*3/2):],thr=reject_thr)
    epoch_post_early = reject_crazy_epoch(epoch['SWS_Post'][:int(len(epoch)/7*3/2)],thr=reject_thr)
    epoch_post_late = reject_crazy_epoch(epoch['SWS_Post'][int(len(epoch)/7*3/2):],thr=reject_thr)

    epoch_pre_early_list = epoch_pre_early_list + [epoch_pre_early]
    epoch_pre_late_list = epoch_pre_late_list + [epoch_pre_late]
    epoch_post_early_list = epoch_post_early_list + [epoch_post_early]
    epoch_post_late_list = epoch_post_late_list + [epoch_post_late]

    evk_pre_early_list = evk_pre_early_list + [epoch_pre_early.average()]
    evk_pre_late_list = evk_pre_late_list + [epoch_pre_late.average()]
    evk_post_early_list = evk_post_early_list + [epoch_post_early.average()]
    evk_post_late_list = evk_post_late_list + [epoch_post_late.average()]
    
    pre_evk_list = pre_evk_list + [reject_crazy_epoch(epoch['SWS_Pre'],thr=reject_thr).average()]
    post_evk_list = post_evk_list + [reject_crazy_epoch(epoch['SWS_Post'],thr=reject_thr).average()]
    reject_crazy_epoch(epoch['SWS_Pre'],thr=reject_thr).average().plot_joint()
    print('--',i,'-----------Pre-----------\n\n')
    reject_crazy_epoch(epoch['SWS_Post'],thr=reject_thr).average().plot_joint()
    print('--',i,'-----------Post-----------\n\n')
    print('////////////////////////////////////////////////////////////////////////////////////////////////////////////////\n\n')

    pre_evk_list_c1 = pre_evk_list_c1 + [reject_crazy_epoch(epoch_c1['SWS_Pre'],thr=reject_thr).average()]
    post_evk_list_c1 = post_evk_list_c1 + [reject_crazy_epoch(epoch_c1['SWS_Post'],thr=reject_thr).average()]

    pre_evk_list_c2 = pre_evk_list_c2 + [reject_crazy_epoch(epoch_c2['SWS_Pre'],thr=reject_thr).average()]
    post_evk_list_c2 = post_evk_list_c2 + [reject_crazy_epoch(epoch_c2['SWS_Post'],thr=reject_thr).average()]

pre_avg = mne.grand_average(pre_evk_list)
post_avg = mne.grand_average(post_evk_list)
pre_avg_c1 = mne.grand_average(pre_evk_list_c1)
post_avg_c1 = mne.grand_average(post_evk_list_c1)
pre_avg_c2 = mne.grand_average(pre_evk_list_c2)
post_avg_c2 = mne.grand_average(post_evk_list_c2)

In [ ]:
times = np.arange(0, 0.8, 0.1)

In [ ]:
pre_avg.plot(spatial_colors=True)
pre_avg.plot_image()
pre_avg.plot_topomap(times,average=0.1,extrapolate = 'local',vlim=(-1.5, 1.5))

In [ ]:
post_avg.plot(spatial_colors=True)
post_avg.plot_image()
post_avg.plot_topomap(times,average=0.1,extrapolate = 'local',vlim=(-1.5, 1.5))

In [ ]:
diff_avg = mne.combine_evoked([pre_avg,post_avg],weights=[1,-1])

In [ ]:
diff_avg.plot(spatial_colors=True)
diff_avg.plot_image()

In [ ]:
diff_avg.plot_topomap(times,average=0.1,extrapolate = 'local',vlim=(-1.5, 1.5))

In [ ]:
ch_adjacency,_ = mne.channels.find_ch_adjacency(epoch.info,ch_type = 'eeg')

# Pre / Post Ttest

In [ ]:
def stat_fun(*args):
    stat = ttest(args[0],args[1])
    return stat.statistic
alpha = 0.05
n_permutations = 5000
dof = n_participants-1
t_threshold_lower = stats.t.ppf(alpha / 2, dof)
t_threshold_upper = stats.t.ppf(1 - alpha / 2, dof)
print(f"Two-tailed t-value thresholds: {t_threshold_lower:.3f} and {t_threshold_upper:.3f}")
t_thresh = t_threshold_upper

In [ ]:
data_shape = evk_pre_early_list[0].get_data().shape
evk_dataframe = np.zeros([n_participants,2,data_shape[0],data_shape[1]])
for i in range(len(evk_pre_early_list)):
    evk_dataframe[i,0,:,:] = (evk_pre_early_list[i].get_data() + evk_pre_late_list[i].get_data())/2
    evk_dataframe[i,1,:,:] = (evk_post_early_list[i].get_data() + evk_post_late_list[i].get_data())/2

In [ ]:
result = ttest(evk_dataframe[:,0,:,:],evk_dataframe[:,1,:,:],)
effect = result.statistic
sig = result.pvalue
#df = result.df
#confidence_interval = result.df

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 2))
ax.imshow(
        effect,
        cmap="gray",
        aspect="auto",
        origin="lower",
        extent=[-0.2, 1, 0, 64],
        vmin=-5, vmax=5
    )

effect[sig >= 0.05] = np.nan
c = ax.imshow(
        effect,
        cmap="coolwarm",
        aspect="auto",
        origin="lower",
        extent=[-0.2, 1, 0, 64],
        vmin=-5, vmax=5
    )

#cbar = fig.colorbar(c, ax=ax)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Channel")
ax.set_title('Time-locked response')
fig.colorbar(c, ax=ax,ticks=[-5,0,5])

In [ ]:
pre_evk_list = []
post_evk_list = []
for i in range (11):
    epoch_file_path = get_epoch_path(raw_eeg_path,i,blk_list[1],type_list[1])
    epoch = mne.read_epochs(epoch_file_path)
    pre_evk_list = pre_evk_list + [epoch['SWS_Pre'].average()]
    post_evk_list = post_evk_list + [epoch['SWS_Post'].average()]

pre_avg = mne.grand_average(pre_evk_list)
post_avg = mne.grand_average(post_evk_list)

In [ ]:
data = evk_dataframe
window_data = data[:,:,:,window]
T_obs, _, _, _ = mne.stats.spatio_temporal_cluster_test(
    np.swapaxes(np.swapaxes(np.asarray(data), 0, 1),2,3),
    stat_fun=stat_fun,
    threshold=t_thresh,
    tail=0,
    n_jobs=None,
    n_permutations=1,
    buffer_size=None,
    out_type="mask",
    adjacency=ch_adjacency,
    t_power=1,
    step_down_p=0.05
)
T_obs_window, clusters, cluster_p_values, h0 = mne.stats.spatio_temporal_cluster_test(
    np.swapaxes(np.swapaxes(np.asarray(window_data), 0, 1),2,3),
    stat_fun=stat_fun,
    threshold=t_thresh,
    tail=0,
    n_jobs=None,
    n_permutations=n_permutations,
    buffer_size=None,
    out_type="mask",
    adjacency=ch_adjacency,
    t_power=1,
    step_down_p=0.05
)

In [ ]:
cluster_p_values

In [ ]:
good_clusters = np.where(cluster_p_values < 0.05)
print(good_clusters)
T_obs_plot = T_obs.copy()

if good_clusters[0].shape[0]>1:
    target_clusters_window = np.sum(np.array(clusters)[np.squeeze(good_clusters)],0)>0
elif good_clusters[0].shape[0]==1:
    target_clusters_window = np.array(clusters)[np.squeeze(good_clusters)]>0
elif good_clusters[0].shape[0]<1:
    target_clusters_window = 0*np.array(clusters)[0,:,:]!=0
    
target_clusters = np.zeros(T_obs_plot.shape).astype('bool')
print(target_clusters.shape,target_clusters_window.shape)
target_clusters[window,:] = target_clusters_window
        
T_obs_plot[~target_clusters] = np.nan

fig, ax = plt.subplots(figsize=(6, 4))
for f_image, cmap in zip([T_obs.T, T_obs_plot.T], ["gray", "cool"]):
    c = ax.imshow(
        f_image,
        cmap=cmap,
        aspect="auto",
        origin="lower",
        extent=[-0.2, 1, 0, 63],
        vmin=-10, vmax=10
    )

fig.colorbar(c, ax=ax,ticks=[-10,0,10])
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Channel")
ax.set_title(
    f'Pre vs Post"\n'
    "Phase by cluster-level corrected (p <= 0.05)"
)
#ax.spines['top'].set_visible(False) #去掉上边框
#ax.spines['bottom'].set_visible(False) #去掉下边框
#ax.spines['left'].set_visible(False) #去掉左边框
#ax.spines['right'].set_visible(False) #去掉右边框

fig.tight_layout()
plt.show()

In [ ]:
print(T_obs.shape)
print(T_obs_plot.shape)

fig1 = multi_window_topo_plot(-T_obs.T,epoch_all.info,vmin=-4,vmax=4,vmiddle=0,mask=T_obs_plot.T,window_size=0.05,plot_time=0,markersize=1,
                              plot_type='T stat',colormap='RdBu_r',channel_omit = None,n_row=2,n_col=8,fontsize=10,sensors='k.',
                              extrapolate='local',if_threshold=True)

In [ ]:
ch_names_array = np.array(epoch_all.ch_names)

# TFR Early & Late / Pre & Post ANOVA

In [ ]:
TFR_ch_index = np.array([14,21,24,25,49,50,63])
print(ch_names_array[TFR_ch_index])

In [ ]:
def pick_data_ch_time(data,ch_list,time):
    data1 = data.copy()[ch_list,:,:]
    data2 = data1[:,:,time]
    return data2

def generate_tfr_dataframe(epoch_list,tfr_kwargs,TFR_ch_index,tfr_window,tfr_sample_freq):
    data_shape = np.mean(pick_data_ch_time(np.mean(epoch_list[0].compute_tfr(**tfr_kwargs).data,axis=0),TFR_ch_index,tfr_window),axis=0).shape
    tfr_dataframe = np.zeros([len(epoch_list),data_shape[0],data_shape[1]])
    for i,epoch in enumerate(epoch_list):
        tfr = np.mean(epoch.copy().subtract_evoked().compute_tfr(**tfr_kwargs).data,axis=0)
        tfr_dataframe[i,:,:] = np.mean(pick_data_ch_time(tfr,TFR_ch_index,tfr_window),axis=0)
    return tfr_dataframe

In [ ]:
#print(epoch_pre_early_list[0].compute_tfr(**tfr_kwargs).data.shape)
#tfr = np.mean(pick_data_ch_time(np.mean(epoch_pre_early_list[0].compute_tfr(**tfr_kwargs).data,axis=0),TFR_ch_index,tfr_window),axis=0)
print(len(epoch_pre_early_list))

In [ ]:
decim = 2
freqs = np.arange(5, 46, 1)  # define frequencies of interest
print(freqs)
n_cycles = 2.5
tfr_kwargs = dict(
    method="morlet",
    freqs=freqs,
    tmin = 0,
    tmax = 0.8,
    n_cycles=n_cycles,
    decim=decim,
    n_jobs = -1,
    use_fft=False,
)

In [ ]:
t_window = [0,0.8]
tfr_sample_freq = 200 # max freq = 45
tfr_window = np.arange(int(tfr_sample_freq*(t_window[0]+0.2)),int(tfr_sample_freq*(t_window[1]+0.2)))

In [ ]:
data_shape = np.mean(pick_data_ch_time(evk_pre_early_list[0].compute_tfr(**tfr_kwargs).data,TFR_ch_index,tfr_window),axis=0).shape
evk_dataframe = np.zeros([4,n_participants,data_shape[0],data_shape[1]])
print(data_shape)

In [ ]:
evk_dataframe[0,:,:,:] = generate_tfr_dataframe(epoch_pre_early_list,tfr_kwargs,TFR_ch_index,tfr_window,tfr_sample_freq)
evk_dataframe[1,:,:,:] = generate_tfr_dataframe(epoch_pre_late_list,tfr_kwargs,TFR_ch_index,tfr_window,tfr_sample_freq)
evk_dataframe[2,:,:,:] = generate_tfr_dataframe(epoch_post_early_list,tfr_kwargs,TFR_ch_index,tfr_window,tfr_sample_freq)
evk_dataframe[3,:,:,:] = generate_tfr_dataframe(epoch_post_late_list,tfr_kwargs,TFR_ch_index,tfr_window,tfr_sample_freq)

In [ ]:
evk_dataframe = np.swapaxes(evk_dataframe, 0, 1)

In [ ]:
factor_levels = [2, 2]
effects = "A*B"
fvals, pvals = f_mway_rm(evk_dataframe, factor_levels, effects=effects)
print(fvals.shape,pvals.shape)

In [ ]:
effect_labels = [ "Pre vs Post", "Early vs Late", "Interaction"]

fig, axes = plt.subplots(3, 1, figsize=(6, 6))

# let's visualize our effects by computing f-images
for effect, sig, effect_label, ax in zip(fvals, pvals, effect_labels, axes):
    # show naive F-values in gray
    ax.imshow(
        effect,
        cmap="gray",
        aspect="auto",
        origin="lower",
        extent=[0, 0.8, 5, 45],
    )
    # create mask for significant time-frequency locations
    effect[sig >= 0.05] = np.nan
    c = ax.imshow(
        effect,
        cmap="autumn",
        aspect="auto",
        origin="lower",
        extent=[0, 0.8, 5, 45],
    )
    cbar = fig.colorbar(c, ax=ax)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency")
    ax.set_title(f'Time-locked response for "{effect_label}" ')
    c.set_clim(0, 30)

fig.tight_layout()
plt.show()

In [ ]:
def stat_fun(*args):
    return f_mway_rm(
        np.swapaxes(args, 1, 0),
        factor_levels=factor_levels,
        effects=effects,
        return_pvals=False,
    )[0]

In [ ]:
n_permutations = 5000

In [ ]:
# The ANOVA returns a tuple f-values and p-values, we will pick the former.
effects = "A"
pthresh = 0.05  # set threshold rather high to save some time
f_thresh = f_threshold_mway_rm(n_participants, factor_levels, effects, pthresh)
print(f_thresh)
tail = 1  # f-test, so tail > 0

data = evk_dataframe
window_data = data
F_obs, _, _, _ = mne.stats.spatio_temporal_cluster_test(
    np.swapaxes(np.swapaxes(np.asarray(data), 0, 1),2,3),
    stat_fun=stat_fun,
    threshold=f_thresh,
    tail=tail,
    n_jobs=None,
    n_permutations=1,
    buffer_size=None,
    out_type="mask",
    t_power=1,
    step_down_p=0.05
)
F_obs_window, clusters, cluster_p_values, h0 = mne.stats.spatio_temporal_cluster_test(
    np.swapaxes(np.swapaxes(np.asarray(window_data), 0, 1),2,3),
    stat_fun=stat_fun,
    threshold=f_thresh,
    tail=tail,
    n_jobs=None,
    n_permutations=n_permutations,
    buffer_size=None,
    out_type="mask",
    t_power=1,
    step_down_p=0.05
)
print(cluster_p_values)

In [ ]:
good_clusters = np.where(cluster_p_values <= 0.05)
print(good_clusters)
F_obs_plot = F_obs.copy()

if good_clusters[0].shape[0]>1:
    target_clusters_window = np.sum(np.array(clusters)[np.squeeze(good_clusters)],0)>0
elif good_clusters[0].shape[0]==1:
    target_clusters_window = np.array(clusters)[np.squeeze(good_clusters)]>0
elif good_clusters[0].shape[0]<1:
    target_clusters_window = 0*np.array(clusters)[0,:,:]!=0
    
target_clusters = np.zeros(F_obs_plot.shape).astype('bool')
print(target_clusters.shape,target_clusters_window.shape)
target_clusters = target_clusters_window
        
F_obs_plot[~target_clusters] = np.nan

fig, ax = plt.subplots(figsize=(6, 2))
for f_image, cmap in zip([F_obs.T, F_obs_plot.T], ["gray", "cool"]):
    c = ax.imshow(
        f_image,
        cmap=cmap,
        aspect="auto",
        origin="lower",
        extent=[0, 0.8, 5, 45],
        vmin=-10, vmax=10
    )

fig.colorbar(c, ax=ax,ticks=[-10,0,10])
ax.set_yticks([10,20,30,40])
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Frequency")
ax.set_title(
    f'Pre vs Post\n'
    "Cluster-level corrected (p <= 0.05)"
)

fig.tight_layout()
plt.show()

In [ ]:
fig.savefig(fig_exp_path+'Cluster_Pre_vs_Post_TFR.svg')

In [ ]:
# The ANOVA returns a tuple f-values and p-values, we will pick the former.
effects = "B"
pthresh = 0.05  # set threshold rather high to save some time
f_thresh = f_threshold_mway_rm(n_participants, factor_levels, effects, pthresh)
print(f_thresh)
tail = 1  # f-test, so tail > 0

data = evk_dataframe
window_data = data
F_obs, _, _, _ = mne.stats.spatio_temporal_cluster_test(
    np.swapaxes(np.swapaxes(np.asarray(data), 0, 1),2,3),
    stat_fun=stat_fun,
    threshold=f_thresh,
    tail=tail,
    n_jobs=None,
    n_permutations=1,
    buffer_size=None,
    out_type="mask",
    t_power=1,
    step_down_p=0.05
)
F_obs_window, clusters, cluster_p_values, h0 = mne.stats.spatio_temporal_cluster_test(
    np.swapaxes(np.swapaxes(np.asarray(window_data), 0, 1),2,3),
    stat_fun=stat_fun,
    threshold=f_thresh,
    tail=tail,
    n_jobs=None,
    n_permutations=n_permutations,
    buffer_size=None,
    out_type="mask",
    t_power=1,
    step_down_p=0.05,
    seed = 9
)
print(cluster_p_values)

In [ ]:
good_clusters = np.where(cluster_p_values <= 0.05)
print(good_clusters)
F_obs_plot = F_obs.copy()

if good_clusters[0].shape[0]>1:
    target_clusters_window = np.sum(np.array(clusters)[np.squeeze(good_clusters)],0)>0
elif good_clusters[0].shape[0]==1:
    target_clusters_window = np.array(clusters)[np.squeeze(good_clusters)]>0
elif good_clusters[0].shape[0]<1:
    target_clusters_window = 0*np.array(clusters)[0,:,:]!=0
    
target_clusters = np.zeros(F_obs_plot.shape).astype('bool')
print(target_clusters.shape,target_clusters_window.shape)
target_clusters = target_clusters_window
        
F_obs_plot[~target_clusters] = np.nan

fig, ax = plt.subplots(figsize=(6, 2))
for f_image, cmap in zip([F_obs.T, F_obs_plot.T], ["gray", "cool"]):
    c = ax.imshow(
        f_image,
        cmap=cmap,
        aspect="auto",
        origin="lower",
        extent=[0, 0.8, 5, 45],
        vmin=-10, vmax=10
    )

fig.colorbar(c, ax=ax,ticks=[-10,0,10])
ax.set_yticks([10,20,30,40])
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Frequency")
ax.set_title(
    f'Early vs Late\n'
    "Cluster-level corrected (p <= 0.05)"
)

fig.tight_layout()
plt.show()

In [ ]:
fig.savefig(fig_exp_path+'Cluster_Early_vs_Late_TFR.svg')

In [ ]:
# The ANOVA returns a tuple f-values and p-values, we will pick the former.
effects = "A:B"
pthresh = 0.05  # set threshold rather high to save some time
f_thresh = f_threshold_mway_rm(n_participants, factor_levels, effects, pthresh)
print(f_thresh)
tail = 1  # f-test, so tail > 0

data = evk_dataframe
window_data = data
F_obs, _, _, _ = mne.stats.spatio_temporal_cluster_test(
    np.swapaxes(np.swapaxes(np.asarray(data), 0, 1),2,3),
    stat_fun=stat_fun,
    threshold=f_thresh,
    tail=tail,
    n_jobs=None,
    n_permutations=1,
    buffer_size=None,
    out_type="mask",
    t_power=1,
    step_down_p=0.05
)
F_obs_window, clusters, cluster_p_values, h0 = mne.stats.spatio_temporal_cluster_test(
    np.swapaxes(np.swapaxes(np.asarray(window_data), 0, 1),2,3),
    stat_fun=stat_fun,
    threshold=f_thresh,
    tail=tail,
    n_jobs=None,
    n_permutations=n_permutations,
    buffer_size=None,
    out_type="mask",
    t_power=0,
    step_down_p=0
)
print(cluster_p_values)

In [ ]:
good_clusters = np.where(cluster_p_values <= 0.05)
#good_clusters = np.where(cluster_p_values >= min(cluster_p_values))
print(good_clusters)
F_obs_plot = F_obs.copy()

if good_clusters[0].shape[0]>1:
    target_clusters_window = np.sum(np.array(clusters)[np.squeeze(good_clusters)],0)>0
elif good_clusters[0].shape[0]==1:
    target_clusters_window = np.array(clusters)[np.squeeze(good_clusters)]>0
elif good_clusters[0].shape[0]<1:
    target_clusters_window = 0*np.array(clusters)[0,:,:]!=0
    
target_clusters = np.zeros(F_obs_plot.shape).astype('bool')
print(target_clusters.shape,target_clusters_window.shape)
target_clusters = target_clusters_window
        
F_obs_plot[~target_clusters] = np.nan

fig, ax = plt.subplots(figsize=(6, 2))
for f_image, cmap in zip([F_obs.T, F_obs_plot.T], ["gray", "cool"]):
    c = ax.imshow(
        f_image,
        cmap=cmap,
        aspect="auto",
        origin="lower",
        extent=[0, 0.8, 5, 45],
        vmin=-10, vmax=10
    )

fig.colorbar(c, ax=ax,ticks=[-10,0,10])
ax.set_yticks([10,20,30,40])
ax.set_xlabel("Time (s)")
ax.set_ylabel("Frequency")
ax.set_title(
    f'Interaction\n'
    "Cluster-level corrected (p <= 0.05)"
)

fig.tight_layout()
plt.show()

In [ ]:
fig.savefig(fig_exp_path+'Cluster_Interaction_TFR.svg')